# 03 · Point-in-Time 與時間穿越示範（leakage demo）

示範「為什麼滾動特徵要先 `shift(1)`」：若直接對含當期值的視窗聚合，
特徵會偷看到 *t* 時刻的觀測，造成 **時間穿越（leakage）**，離線指標虛高、
上線後崩盤。本 notebook 對比「洩漏版 vs 正確版」特徵。

In [ ]:
from src.data.loaders import load_sensors
from src.features.build_features import build_sensor_features

df = load_sensors()
# 正確版：build_sensor_features 內部已 shift(1)，t 只看得到 t-1 以前。
correct = build_sensor_features(df, window=3)
correct[["machine_id", "event_timestamp", "temperature", "temp_roll_mean"]].head(6)

In [ ]:
# 洩漏版（反例）：未 shift，rolling 視窗包含當期 temperature → 偷看未來。
leaky = df.sort_values(["machine_id", "event_timestamp"]).copy()
leaky["temp_roll_mean_LEAKY"] = (
    leaky.groupby("machine_id")["temperature"]
    .rolling(window=3, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)
leaky[["machine_id", "event_timestamp", "temperature", "temp_roll_mean_LEAKY"]].head(6)

In [ ]:
# 對比：洩漏版第一列已含當期值（=temperature 本身），正確版第一列為中性 0。
import pandas as pd

cmp = pd.DataFrame({
    "temperature": df.sort_values(["machine_id", "event_timestamp"])["temperature"].values[:5],
    "correct(shift=1)": correct["temp_roll_mean"].values[:5],
    "leaky(no shift)": leaky["temp_roll_mean_LEAKY"].values[:5],
})
cmp

**結論**：正確版第一列特徵為 0（尚無歷史），洩漏版第一列直接等於當期溫度。
上線時你拿不到「當期之後」的資料，所以洩漏版的離線分數是假的。
Feast 的 `get_historical_features` 以 `event_timestamp` 做 as-of join，
從機制上保證 point-in-time 正確。